In [1]:
import os
import psutil
import glob
import time
import math
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from datasets import Dataset
from scipy.stats import norm

#from sklearn.metrics import accuracy_score
#from sklearn.naive_bayes import GaussianNB
#from sklearn.preprocessing import LabelEncoder
#from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

#from sklearn.model_selection import GridSearchCV, StratifiedKFold, LeaveOneOut, RandomizedSearchCV
#from sklearn.ensemble import RandomForestClassifier
#from sklearn.datasets import make_classification
#from sklearn.model_selection import train_test_split
#from sklearn.metrics import accuracy_score, f1_score

#from transformers import Trainer, TrainingArguments
#from transformers import BertTokenizer
#from transformers import BertForSequenceClassification, TrainingArguments, Trainer
#from transformers import AutoModel, AutoTokenizer
#from transformers.models.mamba import MambaForCausalLM, MambaConfig

#from langdetect import detect
#from langdetect.lang_detect_exception import LangDetectException
#from transformers import AutoModelForSequenceClassification
#from transformers import TextClassificationPipeline


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [2]:
def Validate_Folder( sl_aux_000 ):
    try:	
        os.stat( sl_aux_000 )
        return True
    except:	
        os.mkdir( sl_aux_000 )
        return False

In [3]:
os.environ["WANDB_DISABLED"] = "true"
os.chdir("D:/MAESTRIA/5_CUATRIMESTRE/PROYECTO_LENGUAJE/SEMANA_05")    # PC
# os.chdir("E:/MAESTRIA/5_CUATRIMESTRE/PROYECTO_LENGUAJE/SEMANA_05")  # Laptop
os.getcwd()             ## Verificacion del directorio actual.

'D:\\MAESTRIA\\5_CUATRIMESTRE\\PROYECTO_LENGUAJE\\SEMANA_05'

In [4]:
# Preprocess text (customize as needed)
def preprocess(text):
    return text.lower().strip()  # Add more steps (e.g., remove stopwords, lemmatize)

## Datasets

In [8]:
# HATEXPLAIN: 20148 elements.
df0_HATEXPLAIN = pd.read_csv('./HATEXPLAIN/HATEXPLAIN_CLEAN_True_Final_01.csv')
df0_HATEXPLAIN = df0_HATEXPLAIN.rename(columns={'post_tokens': 'comment', 'hate_speech_flag': 'isHate'})
df0_HATEXPLAIN["clean_text"] = df0_HATEXPLAIN["comment"].apply(preprocess)
df1_HATEXPLAIN = df0_HATEXPLAIN[['clean_text', 'isHate']]

# ETHOS: 997 elements.
df0_ETHOS = pd.read_csv('./ETHOS/Ethos_Dataset_Binary.csv', sep=";")
df0_ETHOS["clean_text"] = df0_ETHOS["comment"].apply(preprocess)
df1_ETHOS = df0_ETHOS[['clean_text', 'isHate']]
df1_ETHOS['isHate'] = df1_ETHOS['isHate'].astype(int)

# Concat the 2 dataframes in one.
df_2_Datasets = pd.concat([df1_HATEXPLAIN, df1_ETHOS], ignore_index=True)

C:\Users\sebas\AppData\Local\Temp\ipykernel_30612\2104760022.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1_ETHOS['isHate'] = df1_ETHOS['isHate'].astype(int)


In [10]:
# Tokenization function using BERT tokenizer
def tokenize(batch):
    return tokenizer(batch['Text'], padding="max_length", truncation=True, max_length=512)

# Use of HateBert:
tokenizer = AutoTokenizer.from_pretrained("GroNLP/hateBERT")
model = AutoModelForSequenceClassification.from_pretrained("GroNLP/hateBERT")

# Split the dataset into training and testing sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_2_Datasets['clean_text'], 
    df_2_Datasets['isHate'], 
    test_size=0.2, 
    random_state=42
)

# Create Hugging Face datasets from your split texts and labels
train_dataset = Dataset.from_dict({
    "Text": train_texts.tolist(), 
    "labels": train_labels.tolist()
})
test_dataset = Dataset.from_dict({
    "Text": test_texts.tolist(), 
    "labels": test_labels.tolist()
})

# Apply tokenization to the datasets
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Remove the original text column, keeping only tokenized inputs and labels
train_dataset = train_dataset.remove_columns("Text")
test_dataset = test_dataset.remove_columns("Text")

# Set dataset format to PyTorch tensors
train_dataset.set_format("torch")
test_dataset.set_format("torch")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/hateBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/16916 [00:00<?, ? examples/s]

Map:   0%|          | 0/4230 [00:00<?, ? examples/s]

## Preprocessing of the selected videos:

In [ ]:
csv_Files = glob.glob(os.path.join("Youtube_Videos//", "*.csv"))
dataframes = []
for csv in csv_Files:
    try:
        df_temp = pd.read_csv(csv)
        dataframes.append(df_temp)
        print(f"Cargado: {csv}")
    except Exception as e:
        print(f"Error al cargar {csv}: {e}")

In [ ]:
# Fuction to detect english language.
def detect_english(texto):
    try:
        return detect(str(texto)) == 'en'
    except LangDetectException:
        return False

# Filter only the english language detected columns.
dataframes_english = []

for df in dataframes:
    dataframes_english.append( df[df["text"].apply(detect_english)] )
dataframes_english[0]   

# Save the english filtration dataframes. 
Validate_Folder( "English_Filter" )

for df in dataframes_english:
    aux_00 = df['video_title'].unique()[0]
    df.to_csv("./English_Filter/" + aux_00 + "_en" + ".csv", index=False)

In [12]:
# Read the language filtered files
csv_Files = glob.glob(os.path.join("English_Filter//", "*.csv"))
dataframes_english = []
for csv in csv_Files:
    try:
        df_temp = pd.read_csv(csv)
        dataframes_english.append(df_temp)
        print(f"Cargado: {csv}")
    except Exception as e:
        print(f"Error al cargar {csv}: {e}")

Cargado: English_Filter\Black Sabbath ~ War Pigs_en.csv
Cargado: English_Filter\Hallowed Be Thy Name (2015 Remaster)_en.csv
Cargado: English_Filter\Master of Puppets (Remastered)_en.csv
Cargado: English_Filter\Pantera - Cowboys From Hell (Official Music Video) [4K Remaster]_en.csv
Cargado: English_Filter\Slayer - Raining Blood_en.csv


## Transformers Model

In [13]:
# Initialize the BERT model for sequence classification
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Use of HateBert:
tokenizer = AutoTokenizer.from_pretrained("GroNLP/hateBERT")
model = AutoModelForSequenceClassification.from_pretrained("GroNLP/hateBERT")

# Ensure model is on GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/hateBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [127]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./bert-hate-speech",
    evaluation_strategy="epoch",
    save_strategy="no",  # Disable checkpoint saving
    learning_rate=2e-5,
    ##########################################################################
    # Adjust based on GPU memory ( RTX 4090 )
    ##########################################################################
    per_device_train_batch_size=64,     # Max batch size for RTX 4090
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=1,
    dataloader_num_workers=1,
    fp16=True,  # Use half-precision
    bf16=False,
    ##########################################################################
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=False,  # Disable loading best model at the end
    metric_for_best_model="accuracy",
    report_to="none",
    save_total_limit=0,  # Optional: Ensures no checkpoints are kept
)

c:\Users\sebas\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [128]:
# Define the metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = torch.argmax(torch.tensor(predictions), axis=1).numpy()
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

In [129]:
# Create the Trainer
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,   # Assumes train_dataset is already prepared
    eval_dataset    = test_dataset,       # Assumes test_dataset is already prepared
    tokenizer       = tokenizer,             # The corresponding tokenizer (e.g., BertTokenizer)
    compute_metrics = compute_metrics,
)

C:\Users\sebas\AppData\Local\Temp\ipykernel_29420\614990831.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [130]:
trainer.train()

# ( per_device_train_batch_size , per_device_eval_batch_size , gradient_accumulation_steps , epochs )
# ( 2 , 2 , 1 , 30 )        :==: 2:47:05  
# ( 2 , 2 , 2 , 3 )         :==: 16.57 it/s
# ( 1 , 1 , 1 , 3 )         :==: 28.03 it/s
## ( per_device_train_batch_size , per_device_eval_batch_size , gradient_accumulation_steps , dataloader_num_workers , epochs )
# ( 16 , 16 , 1 , 8 , 3 )   :==: N/A
# ( 16 , 16 , 1 , 4 , 3 )   :==: N/A
# ( 16 , 16 , 2 , 1 , 3 )   :==: N/A - [ 187/1512 00:25 < 03:02, 7.27 it/s, Epoch 0.37/3]
# ( 16 , 16 , 1 , 1 , 3 )   :==: N/A - [ 913/3024 01:07 < 02:37, 13.44 it/s, Epoch 0.90/3]
# ( 32 , 16 , 1 , 1 , 3 )   :==: N/A - [ 217/1512 00:29 < 02:59, 7.22 it/s, Epoch 0.43/3]
# ( 64 , 64 , 1 , 1 , 3 )   :==: N/A - [ 130/756 00:34 < 02:48, 3.71 it/s, Epoch 0.51/3] ## Optimal configuration based on the selected parameters. 
# ( 64 , 64 , 1 , 2 , 3 )   :==: N/A - [ 35/756 00:24 < 08:45, 1.37 it/s, Epoch 0.13/3]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.371200,0.349631,0.854137,0.742810
2,0.305900,0.355594,0.856501,0.741812
3,0.232400,0.391437,0.852719,0.730420
4,0.163900,0.455876,0.846809,0.736585
5,0.126700,0.512234,0.837825,0.732449
6,0.077300,0.607946,0.837589,0.704770
7,0.058500,0.678382,0.834515,0.722443
8,0.038700,0.743262,0.833570,0.717723
9,0.032800,0.765250,0.835934,0.710592
10,0.028600,0.787729,0.838534,0.721111


TrainOutput(global_step=2650, training_loss=0.14719411229187587, metrics={'train_runtime': 827.8184, 'train_samples_per_second': 204.344, 'train_steps_per_second': 3.201, 'total_flos': 4.45078661246976e+16, 'train_loss': 0.14719411229187587, 'epoch': 10.0})

In [131]:
results = trainer.evaluate()
print("Evaluation results:", results)

Evaluation results: {'eval_loss': 0.7877289652824402, 'eval_accuracy': 0.8385342789598109, 'eval_f1': 0.7211106574111883, 'eval_runtime': 8.3127, 'eval_samples_per_second': 508.862, 'eval_steps_per_second': 8.06, 'epoch': 10.0}


In [132]:
# Prediction function using the fine-tuned BERT model
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=1).item()
    return "Hate Speech" if prediction == 1 else "Not Hate Speech"

In [133]:
# Example usage of predict function
text_example = "Example text to evaluate hate speech."
print("Prediction:", predict(text_example))

Prediction: Not Hate Speech


In [134]:
text_example = "martian kike"
print("Prediction:", predict(text_example))

Prediction: Hate Speech


In [137]:
# Save model and tokenizer
model_path = "bert_hate_speech_finetuned_008"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

('bert_hate_speech_finetuned_008\\tokenizer_config.json',
 'bert_hate_speech_finetuned_008\\special_tokens_map.json',
 'bert_hate_speech_finetuned_008\\vocab.txt',
 'bert_hate_speech_finetuned_008\\added_tokens.json',
 'bert_hate_speech_finetuned_008\\tokenizer.json')

## Transformers Workflow

In [138]:
# Later: Load the fine-tuned model and tokenizer
model_path = "./bert_hate_speech_finetuned_008"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(device)

# Creation of the pipeline that uses the fine tuned model.
pipeline = TextClassificationPipeline(  model=model, 
                                        tokenizer=tokenizer, 
                                        return_all_scores=False,
                                        truncation=True,  
                                        device=0 if torch.cuda.is_available() else -1)


def Classify_hate(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0
    resultado = pipeline(text)[0]  # [{'label': 'LABEL_0', 'score': 0.98}]
    # Ajusta el nombre de la etiqueta si es necesario
    return 1 if resultado['label'] in ['hate', 'LABEL_1', '1'] else 0

Device set to use cuda:0
c:\Users\sebas\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\pipelines\text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [139]:
Classify_hate("kike")

1

In [141]:
# Use the Fine tuned model to detect hate speech by creating a new column is_Hate?
aux_model_00 = "Hate_Speech_008"
Validate_Folder( aux_model_00 )
aux_00 = ""

for df in dataframes_english:
    aux_00 = df['video_title'].unique()[0]
    df['is_Hate?'] = df['text'].apply(Classify_hate)
    df.to_csv("./" + aux_model_00 + "/" + aux_00 + "_is_Hate" + ".csv", index=False)

In [142]:
# Count the quantity of the hate speech detection
print( "Black Sabbath ~ War Pigs : " + str( dataframes_english[0]["is_Hate?"].sum() ) )
print( "Hallowed Be Thy Name (2015 Remaster) : " + str( dataframes_english[1]["is_Hate?"].sum() ) )
print( "Master of Puppets (Remastered) : " + str( dataframes_english[2]["is_Hate?"].sum() ) )
print( "Pantera - Cowboys From Hell (Official Music Video) [4K Remaster] : " + str( dataframes_english[3]["is_Hate?"].sum() ) )
print( "Slayer - Raining Blood : " + str( dataframes_english[4]["is_Hate?"].sum() ) )

Black Sabbath ~ War Pigs : 229
Hallowed Be Thy Name (2015 Remaster) : 3
Master of Puppets (Remastered) : 7
Pantera - Cowboys From Hell (Official Music Video) [4K Remaster] : 39
Slayer - Raining Blood : 111


In [37]:
print( "Black Sabbath ~ War Pigs : " + str( dataframes_english[0]["text"].count() ) )
print( "Hallowed Be Thy Name (2015 Remaster) : " + str( dataframes_english[1]["text"].count() ) )
print( "Master of Puppets (Remastered) : " + str( dataframes_english[2]["text"].count() ) )
print( "Pantera - Cowboys From Hell (Official Music Video) [4K Remaster] : " + str( dataframes_english[3]["text"].count() ) )
print( "Slayer - Raining Blood : " + str( dataframes_english[4]["text"].count() ) )

Black Sabbath ~ War Pigs : 29068
Hallowed Be Thy Name (2015 Remaster) : 1793
Master of Puppets (Remastered) : 10149
Pantera - Cowboys From Hell (Official Music Video) [4K Remaster] : 10724
Slayer - Raining Blood : 17852


## Embbedings

In [ ]:
# Function to get text embeddings using the underlying BERT model
def get_text_embedding(text, model, tokenizer, layer=-1):
    # Tokenize the input text with appropriate padding and truncation
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Forward pass with hidden states enabled.
    # For a sequence classification model, the underlying BERT model is available as `model.bert`
    with torch.no_grad():
        outputs = model.bert(**inputs, output_hidden_states=True)
    
    # Extract hidden states from the specified layer (default: last layer)
    hidden_states = outputs.hidden_states[layer]  # shape: (batch_size, seq_length, hidden_size)
    
    # Pooling: Mean pooling across the sequence length dimension
    embedding = hidden_states.mean(dim=1).squeeze().cpu().numpy()
    return embedding

In [ ]:
# Example usage of get_text_embedding function
text_probe = "This is a probe text to obtain its embedding."
embedding = get_text_embedding(text_probe, model, tokenizer)
print("Embedding (first 100 values):", embedding[:100])

## Mamba Model

In [19]:
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel, MambaConfig

ModuleNotFoundError: No module named 'mamba_ssm'

In [14]:
# Tokenization function using BERT tokenizer
def tokenize(batch):
    return tokenizer(batch['Text'], padding="max_length", truncation=True, max_length=512)

# Use of HateBert:
tokenizer = AutoTokenizer.from_pretrained("GroNLP/hateBERT")
model = AutoModelForSequenceClassification.from_pretrained("GroNLP/hateBERT")

# Split the dataset into training and testing sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_2_Datasets['clean_text'], 
    df_2_Datasets['isHate'], 
    test_size=0.2, 
    random_state=42
)

# Create Hugging Face datasets from your split texts and labels
train_dataset = Dataset.from_dict({
    "Text": train_texts.tolist(), 
    "labels": train_labels.tolist()
})
test_dataset = Dataset.from_dict({
    "Text": test_texts.tolist(), 
    "labels": test_labels.tolist()
})

# Apply tokenization to the datasets
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Remove the original text column, keeping only tokenized inputs and labels
train_dataset = train_dataset.remove_columns("Text")
test_dataset = test_dataset.remove_columns("Text")

# Set dataset format to PyTorch tensors
train_dataset.set_format("torch")
test_dataset.set_format("torch")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/hateBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/16916 [00:00<?, ? examples/s]

Map:   0%|          | 0/4230 [00:00<?, ? examples/s]

In [15]:
config = MambaConfig(vocab_size=60, hidden_size=64, num_hidden_layers=4, use_mambapy=True)
model = MambaForCausalLM(config)

# Ensure model is on GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the mamba.py backend. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


MambaForCausalLM(
  (backbone): MambaModel(
    (embeddings): Embedding(60, 64)
    (layers): ModuleList(
      (0-3): 4 x MambaBlock(
        (norm): MambaRMSNorm(64, eps=1e-05)
        (mixer): MambaMixer(
          (conv1d): Conv1d(128, 128, kernel_size=(4,), stride=(1,), padding=(3,), groups=128)
          (act): SiLU()
          (in_proj): Linear(in_features=64, out_features=256, bias=False)
          (x_proj): Linear(in_features=128, out_features=36, bias=False)
          (dt_proj): Linear(in_features=4, out_features=128, bias=True)
          (out_proj): Linear(in_features=128, out_features=64, bias=False)
        )
      )
    )
    (norm_f): MambaRMSNorm(64, eps=1e-05)
  )
  (lm_head): Linear(in_features=64, out_features=60, bias=False)
)

In [16]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./bert-hate-speech",
    evaluation_strategy="epoch",
    save_strategy="no",  # Disable checkpoint saving
    learning_rate=2e-5,
    ##########################################################################
    # Adjust based on GPU memory ( RTX 4090 )
    ##########################################################################
    per_device_train_batch_size=64,     # Max batch size for RTX 4090
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=1,
    dataloader_num_workers=1,
    fp16=True,  # Use half-precision
    bf16=False,
    ##########################################################################
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=False,  # Disable loading best model at the end
    metric_for_best_model="accuracy",
    report_to="none",
    save_total_limit=0,  # Optional: Ensures no checkpoints are kept
)

c:\Users\sebas\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [17]:
trainer.train()

NameError: name 'trainer' is not defined